In [ ]:
# Re-import edited fund_clustering modules automatically before each cell runs.
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the newest weekly export: allocations + metadata, equity funds only.
import sys
from pathlib import Path

from fund_clustering.load import latest_week, load_week

# Should point into FundClustering\.venv; anything else means the wrong kernel.
print(sys.executable)

DATA_DIR = Path("../data")  # git-ignored copies of the CSV exports
WEEK = latest_week(DATA_DIR)

data = load_week(DATA_DIR, WEEK)

# Week, (funds, columns), country column count, sector column count
WEEK, data.funds.shape, len(data.country_columns), len(data.sector_columns)


In [ ]:
# Cosine features: one vector per fund (countries + sectors), scaled to length 1
# so funds are compared by allocation mix, not size.
import numpy as np

from fund_clustering.features import cosine_features

X = cosine_features(data)

# Sanity check: every row should have length 1 (tiny rounding is fine).
lengths = np.linalg.norm(X, axis=1)

X.shape, lengths.min(), lengths.max()


In [ ]:
# Clustering: 10D UMAP + HDBSCAN gives a cluster id per fund (-1 = not in a
# cluster); a separate 2D UMAP gives the map position. ~20 s on first run.
from fund_clustering.cluster import cluster_funds

result = cluster_funds(X)

# Working table for the cells below: fund data plus cluster and map position.
funds = data.funds.assign(
    cluster=result.labels, x=result.xy[:, 0], y=result.xy[:, 1]
)

# (number of clusters, funds not in a cluster)
result.cluster_count, result.not_in_cluster_count


In [ ]:
# Naming: each cluster is named after its most common category ("Mixed · A / B"
# under 50% purity); each fund gets a verdict comparing its category to that.
from fund_clustering.naming import describe_clusters, fund_verdicts

clusters = describe_clusters(funds["category"], result.labels)
funds["verdict"] = fund_verdicts(funds["category"], result.labels, clusters)
funds["cluster_name"] = funds["cluster"].map(clusters["name"]).fillna("Not in a cluster")

# agree / neighbouring (same regional family) / different (other family) / none
display(funds["verdict"].value_counts())

# Largest clusters first; purity = share of the leading category.
clusters[["name", "size", "purity"]].sort_values("size", ascending=False)


In [ ]:
# Map: one dot per fund, coloured by cluster. Scroll to zoom, drag to pan,
# hover for details. Only local closeness is meaningful, not distances between
# far-apart groups.
import plotly.express as px

# Outline colour by verdict: amber = neighbouring, red = different family.
VERDICT_OUTLINE = {
    "agree": "rgba(0,0,0,0)",
    "neighbouring": "#c58300",
    "different": "#d0353a",
    "none": "rgba(0,0,0,0)",
}

fig = px.scatter(
    funds.assign(cluster=funds["cluster"].astype(str)),  # str -> discrete colours
    x="x",
    y="y",
    color="cluster",
    hover_name="name",
    hover_data=["isin", "category", "cluster_name", "verdict", "company_name"],
    height=750,
)
# plotly makes one trace per cluster; give each trace's dots their outlines.
for trace in fig.data:
    rows = funds["cluster"].astype(str) == trace.name
    trace.marker.line = dict(
        color=funds.loc[rows, "verdict"].map(VERDICT_OUTLINE).tolist(), width=2
    )
fig.update_yaxes(scaleanchor="x")  # same scale on both axes, no stretching
fig.show()
